# BiasUnlearn-style: WarBias training

Run the cells in order to prepare the data, see one optimizer step on CPU, and configure 24 runs: six models in four group/language directions. Enable training separately.

BiasUnlearn combines NPO suppression, anti-stereotype correction and reference KL. The trainer builds adversarial pools and uses development scores to update their direction.

## Setup

Install `requirements-unlearning.txt` and configure the checkpoint and output paths with the [setup guide](SETUP.md). The data and CPU cells do not load model weights.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

repo = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "src/fairforget/unlearning/objectives.py").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from a fair_forget checkout.")
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning import notebook_matrix, objectives, warbias_data  # noqa: E402

print("Algorithm module:", objectives.__name__)
print("PyTorch:", torch.__version__)

METHOD = "biasunlearn"
RUN_TRAINING = False

Algorithm module: fairforget.unlearning.objectives
PyTorch: 2.9.1


## WarBias data

Use the current 1,007 aligned source triplets in Ukrainian and English. Prepare four separate training directions: IDP in Ukrainian, IDP in English, veteran in Ukrainian, and veteran in English. Every direction trains on one group and one language. Both languages remain available for evaluation.

Split whole claim components. Protect every component used by a QA case from training and development. Use one component split for all directions. Choose 20% of eligible components for development, rounded up. Use group and severity labels to keep shared severity levels in both training and development, retain singleton levels in training, and minimize per-group deviations from the requested development fraction. SHA256 resolves ties. Then select one target group and training language for each bundle. Keep the other group out of its training streams. The Ukrainian and English versions keep the same item IDs and splits.

Use complete S sentences as forget targets and complete C sentences as correction targets, with an empty prompt. Use U sentences from the same group and training language for in-domain reference KL. This measures preservation of WarBias text; general capability needs separate evidence.


In [2]:
settings_path = repo / "configs/unlearning/local.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
CONFIGURATION = json.loads((repo / "configs/unlearning/warbias_final.json").read_text())
data_value = (
    os.environ.get("FAIRFORGET_DATA_ROOT")
    or settings.get("data_roots", {}).get("warbias_application")
    or warbias_data.default_root(repo)
)
DATA_ROOT = Path(data_value).expanduser().resolve()
BINDINGS = {}
for group in CONFIGURATION["training_groups"]:
    for language in CONFIGURATION["training_languages"]:
        direction = f"{group}_{language}"
        directory = DATA_ROOT / direction
        if not directory.exists():
            warbias_data.prepare(repo, directory, target_group=group, training_language=language)
        BINDINGS[direction] = warbias_data.validate_bundle(directory, repo)
print("WarBias: 1,007 aligned source triplets per language.")
print(
    json.dumps(
        {name: value["split_counts"] for name, value in BINDINGS.items()},
        ensure_ascii=False,
        indent=2,
    )
)
print("Evaluation languages:", ", ".join(CONFIGURATION["evaluation_languages"]))

WarBias: 1,007 aligned source triplets per language.
{
  "idp_uk": {
    "development": {
      "cases": 3,
      "components": 3,
      "triplets": 20
    },
    "evaluation": {
      "cases": 33,
      "components": 16,
      "triplets": 314
    },
    "training": {
      "cases": 10,
      "components": 7,
      "triplets": 75
    }
  },
  "idp_en": {
    "development": {
      "cases": 3,
      "components": 3,
      "triplets": 20
    },
    "evaluation": {
      "cases": 33,
      "components": 16,
      "triplets": 314
    },
    "training": {
      "cases": 10,
      "components": 7,
      "triplets": 75
    }
  },
  "veteran_uk": {
    "development": {
      "cases": 6,
      "components": 4,
      "triplets": 40
    },
    "evaluation": {
      "cases": 47,
      "components": 18,
      "triplets": 403
    },
    "training": {
      "cases": 21,
      "components": 19,
      "triplets": 155
    }
  },
  "veteran_en": {
    "development": {
      "cases": 6,
      "components"

## CPU example

Synthetic tensors illustrate one optimizer step. Training uses the WarBias rows and settings loaded below.

In [3]:
# Synthetic tensors for one optimizer step on CPU.
torch.manual_seed(42)
forget_logits = torch.nn.Parameter(torch.randn(2, 4, 7))
retain_logits = torch.nn.Parameter(torch.randn(2, 4, 7))
reference_forget = forget_logits.detach().clone()
reference_retain = retain_logits.detach().clone() + 0.1 * torch.randn(2, 4, 7)
labels = torch.tensor([[1, 2, 3, 4], [2, 3, 4, 5]])
completion_mask = torch.tensor([[False, True, True, True], [False, True, True, False]])
retain_mask = torch.tensor([[True, True, True, True], [True, True, True, False]])


def completion_logps(logits):
    return logits.log_softmax(-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)


parameters = [forget_logits, retain_logits]
optimizer = torch.optim.AdamW(parameters, lr=0.01)
optimizer.zero_grad(set_to_none=True)

In [4]:
correction_logits = torch.nn.Parameter(torch.randn(2, 4, 7))
parameters.append(correction_logits)
optimizer = torch.optim.AdamW(parameters, lr=0.01)
optimizer.zero_grad(set_to_none=True)
loss = objectives.biasunlearn_loss(
    completion_logps(forget_logits),
    completion_logps(reference_forget),
    completion_mask,
    completion_logps(correction_logits),
    completion_mask,
    retain_logits,
    reference_retain,
    retain_mask,
    beta=0.1,
    suppression_weight=1.0,
    correction_weight=1.0,
    preservation_weight=1.0,
    correction_reduction="token_mean",
    retain_reduction="token_mean",
)
loss.total.backward()
torch.nn.utils.clip_grad_norm_(parameters, 1.0)
optimizer.step()
direction = objectives.select_biasunlearn_direction(preference=0.6, current="normal")
assert direction == "normal"
print("BiasUnlearn-style CPU optimizer step completed.")
print("The trainer builds adversarial pools and updates their direction on development data.")

BiasUnlearn-style CPU optimizer step completed.
The trainer builds adversarial pools and updates their direction on development data.


## Training

Enable `RUN_TRAINING` to run all 24 model/direction combinations. Every run starts from the pinned base checkpoint. It saves a separate checkpoint, log and receipt under its run ID. Each run uses one GPU with enough memory for the model. Methods use the same rows within each direction.

In [5]:
MODELS_ROOT = os.environ.get("FAIRFORGET_MODELS_ROOT")
output_value = os.environ.get("FAIRFORGET_OUTPUT") or str(
    Path(settings.get("output_root", repo / "outputs/unlearning")) / METHOD
)
OUTPUT = Path(output_value).expanduser().resolve()
PLAN = notebook_matrix.method_plan(
    repo,
    METHOD,
    protocol="warbias_final",
    data_root=DATA_ROOT,
    models_root=MODELS_ROOT,
)
assert len(PLAN) == 24, "Expected six models in each of four training directions."
assert len({item["run_id"] for item in PLAN}) == len(PLAN)
for item in PLAN:
    recipe = item["recipe"]
    print(
        item["run_id"], recipe["model_id"], recipe["model_revision"], "GPUs:", item["required_gpus"]
    )
print("Runs:", len(PLAN), "| Models: 6 | Method:", METHOD, "| Dataset: WarBias")

qwen3_8b__idp_uk Qwen/Qwen3-8B b968826d9c46dd6066d109eabc6255188de91218 GPUs: 1
llama31_8b_instruct__idp_uk meta-llama/Llama-3.1-8B-Instruct 0e9e39f249a16976918f6564b8830bc894c89659 GPUs: 1
gemma3_12b_it__idp_uk google/gemma-3-12b-it 96b6f1eccf38110c56df3a15bffe176da04bfd80 GPUs: 1
lapa12b_instruct__idp_uk lapa-llm/lapa-v0.1.2-instruct 2969fd997f07b33727c6f10795d78ef23cc5c037 GPUs: 1
mamay12b_v2__idp_uk INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v2.0 3cb02013b6b37cfe10f4b3da0fcddc1acef7f244 GPUs: 1
mamay27b_v2__idp_uk INSAIT-Institute/MamayLM-Gemma-3-27B-IT-v2.0 ad1bdfed9646939358adfcba9c88bb4031b50ed7 GPUs: 1
qwen3_8b__idp_en Qwen/Qwen3-8B b968826d9c46dd6066d109eabc6255188de91218 GPUs: 1
llama31_8b_instruct__idp_en meta-llama/Llama-3.1-8B-Instruct 0e9e39f249a16976918f6564b8830bc894c89659 GPUs: 1
gemma3_12b_it__idp_en google/gemma-3-12b-it 96b6f1eccf38110c56df3a15bffe176da04bfd80 GPUs: 1
lapa12b_instruct__idp_en lapa-llm/lapa-v0.1.2-instruct 2969fd997f07b33727c6f10795d78ef23cc5c037 GPUs: 

In [6]:
if RUN_TRAINING:
    from fairforget.unlearning.runtime import verify_runtime

    verify_runtime(require_cuda=False)
    if not torch.cuda.is_available() or torch.cuda.device_count() < max(
        item["required_gpus"] for item in PLAN
    ):
        raise RuntimeError("Allocate one GPU with enough memory for the selected model.")
    # Validate all checkpoints and data before starting the first model.
    recipe_paths = notebook_matrix.prepare_recipes(PLAN, OUTPUT)
    environment = dict(os.environ)
    environment["PYTHONPATH"] = str(repo / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    environment.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    execution = {"method": METHOD, "runs": {}}
    execution_path = OUTPUT / "matrix_execution.json"
    for item, recipe_path in zip(PLAN, recipe_paths, strict=True):
        run_id = item["run_id"]
        command = [
            sys.executable,
            "-m",
            "fairforget.unlearning.trainer",
            "--config",
            str(recipe_path),
            "--output",
            str(OUTPUT / run_id),
        ]
        record = {"status": "running", "recipe": str(recipe_path), "command": command}
        execution["runs"][run_id] = record
        execution_path.write_text(json.dumps(execution, indent=2) + "\n")
        print("Training", run_id, flush=True)
        # A new process releases GPU memory between runs.
        with (OUTPUT / (run_id + ".log")).open("w") as log:
            completed = subprocess.run(
                command,
                cwd=repo,
                env=environment,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=False,
            )
        record.update(
            status="completed" if completed.returncode == 0 else "failed",
            returncode=completed.returncode,
        )
        execution_path.write_text(json.dumps(execution, indent=2) + "\n")
        completed.check_returncode()
    assert len(execution["runs"]) == len(PLAN) == 24
    print("All 24 model-direction runs completed.")
else:
    print("All 24 model-direction runs are configured. Training is disabled.")

All 24 model-direction runs are configured. Training is disabled.


## Evaluation and implementation

Evaluate the same held-out WarBias IDs in Ukrainian and English before and after each run. Keep scores separate by group and evaluation language. A score over the full release is a descriptive diagnostic because it includes training and development rows. All metrics, including severity slices, come from Fair-UK.

The [baseline notebook](../warbias-artifacts/notebooks/warbias_baseline.ipynb) provides the current per-checkpoint metrics. Use the [before/after notebook](evaluate_before_after.ipynb) for native benchmark comparisons. Annotated WarBias triplets use the Fair-UK original/neutral adapter with matched held-out inputs. See the [WarBias workflow](WARBIAS_UNLEARNING.md).

- [Data binding](../src/fairforget/unlearning/warbias_data.py)
- [Loss implementations](../src/fairforget/unlearning/objectives.py)
- [Trainer](../src/fairforget/unlearning/trainer.py)

Each run verifies the pinned model files and training runtime. It checks the merged and reloaded checkpoint on fixed development inputs before marking the export complete. See the [checkpoint rules](WARBIAS_UNLEARNING.md#checkpoint-verification).

Each run saves a checkpoint, loss history and token-exposure log.
